# Figure 1 and Figure S1: panel composition and scale uncertainty

This notebook reproduces the MERFISH, Xenium, and CosMx analyses used for Figure 1 and Figure S1. It preserves the original analysis order and, critically, the backup/restore steps required before objects are temporarily restricted to shared gene sets.

## 0. Setup and paths

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib.colors import LogNorm
from scipy.stats import spearmanr, linregress
from sklearn.neighbors import NearestNeighbors

import kimlabspatial.preprocessing as pp

# Portable repository and deposited-data paths
from scale_aware_st import (
    AgingGeneSourceFiles,
    RepositoryConfig,
    load_figure1d_aging_gene_sets,
    plot_figure1d_venn,
)

config = RepositoryConfig.from_env()
figure_dir = config.results_dir / "figure1"
figure_dir.mkdir(parents=True, exist_ok=True)
internal_data_dir = config.data_dir / "figure1" / "panel_objects"
adata_140_file = internal_data_dir / "adata_140_forfig1.hdf5"
adata_500_file = internal_data_dir / "adata_500_forfig1.hdf5"
adata_1000_file = internal_data_dir / "adata_1000_forfig1.hdf5"

# External Xenium downloads (documented separately; not redistributed)
xenium_root = config.external_data_dir / "xenium"
xenium_replicate_dirs = [
    xenium_root / 'Xenium_V1_FF_Mouse_Brain_MultiSection_1_outs',
    xenium_root / 'Xenium_V1_FF_Mouse_Brain_MultiSection_2_outs',
    xenium_root / 'Xenium_V1_FF_Mouse_Brain_MultiSection_3_outs',
]
xenium_350_dir = xenium_root / 'Xenium_V1_FFPE_wildtype_2_5_months_outs'
xenium_5k_dir = xenium_root / 'Xenium_Prime_Mouse_Brain_Coronal_FF_outs'

# External raw CosMx download (documented separately; not redistributed)
cosmx_dir = config.external_data_dir / "sennet_cosmx" / "SNT638.MWRV.378"

# Deposited matched MERFISH sections used for Fig. 1E-G
spatial_section_dir = config.data_dir / "primary_merfish" / "spatial_sections"
vizgen_section1_dir = spatial_section_dir / "young_anterior_section_example_1"
vizgen_section2_dir = spatial_section_dir / "young_anterior_section_example_2"
ccf_metadata_dir = config.data_dir / "primary_merfish" / "ccf_metadata"

# Set True after downloading the public Xenium and raw SenNet CosMx inputs
# documented in DATA_DOWNLOADS.md. Deposited-data sections run when False.
RUN_PUBLIC_PLATFORM_COMPARISONS = False


## 1. Load and preprocess internal MERFISH datasets

The 500-gene object was originally pooled from four sections. For reproducible figure generation, the notebook reads the saved pooled object directly. The commented block documents how it was created.

In [ ]:
adata_140 = sc.read_h5ad(adata_140_file)
adata_1000 = sc.read_h5ad(adata_1000_file)

# Remove blank probes before calculating panel comparisons.
adata_140 = adata_140[:, ~adata_140.var_names.str.lower().str.contains('blank')].copy()
adata_1000 = adata_1000[:, ~adata_1000.var_names.str.lower().str.contains('blank')].copy()


adata_500 = sc.read_h5ad(adata_500_file)
adata_aepd = adata_500[adata_500.obs['section'] == 'AEPD'].copy()
adata_pvh = adata_500[adata_500.obs['section'] == 'PVH'].copy()


In [ ]:
def qc_and_volume_filter(adata, volume_col):
    """Apply the original nonzero-count and 10th–90th percentile volume/area filters."""
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=(),
        percent_top=None,
        log1p=False,
        inplace=True,
    )
    sc.pp.filter_cells(adata, min_counts=1, inplace=True)
    lower = np.quantile(adata.obs[volume_col], 0.1)
    upper = np.quantile(adata.obs[volume_col], 0.9)
    return adata[
        (adata.obs[volume_col] >= lower)
        & (adata.obs[volume_col] <= upper)
    ].copy()


adata_140 = qc_and_volume_filter(adata_140, 'volume')
adata_500 = qc_and_volume_filter(adata_500, 'volume')
adata_1000 = qc_and_volume_filter(adata_1000, 'volume')
adata_aepd = qc_and_volume_filter(adata_aepd, 'volume')
adata_pvh = qc_and_volume_filter(adata_pvh, 'volume')


### Preserve full post-QC MERFISH objects

The next analysis temporarily subsets all three objects to their common genes. These copies are therefore required and are explicitly restored before pairwise comparisons.

In [ ]:
adata_140_bu = adata_140.copy()
adata_500_bu = adata_500.copy()
adata_1000_bu = adata_1000.copy()


## 2. Fig. 1A and Fig. S1A — MERFISH panel-size comparison

In [ ]:
common_genes= list(set(adata_140.var_names.to_list()) & set(adata_500.var_names.to_list()) & set(adata_1000.var_names.to_list()))
adata_140= adata_140[:, common_genes].copy()
adata_500= adata_500[:, common_genes].copy()
adata_1000= adata_1000[:, common_genes].copy()
sc.pp.calculate_qc_metrics(adata_140, qc_vars=(), percent_top=None, log1p=False, inplace=True)
sc.pp.calculate_qc_metrics(adata_500, qc_vars=(), percent_top=None, log1p=False, inplace=True)
sc.pp.calculate_qc_metrics(adata_1000, qc_vars=(), percent_top=None, log1p=False, inplace=True)


In [ ]:
palette = {"140": "#8ecae6", "500": "#219ebc", "1000": "#023047"}

adata_140.obs["panel_size"] = "140"
adata_500.obs["panel_size"] = "500"
adata_1000.obs["panel_size"] = "1000"

# combine into one dataframe
df = pd.concat([
    adata_140.obs[["total_counts", "panel_size"]],
    adata_500.obs[["total_counts", "panel_size"]],
    adata_1000.obs[["total_counts", "panel_size"]]
])

# basic violin plot
plt.figure(figsize=(6, 4))
sns.violinplot(data=df, x="panel_size", y="total_counts", hue="panel_size", palette=palette, inner="box", cut=0, legend=False)
sns.despine()
plt.xlabel("Panel size (number of genes)")
plt.ylabel("Total transcripts per cell")
plt.title("Total counts per cell vs panel size (common genes only)")
# plt.title("Total counts per cell vs panel size")
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panela_all.png", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
for adata, label in zip([adata_140, adata_500, adata_1000], ["140", "500", "1000"]):
    adata.obs["avg_gene_count"] = adata.obs["total_counts"] / adata.obs["n_genes_by_counts"]
    adata.obs["panel_size"] = label

# for adata, label in zip([adata_140, adata_500], ["140", "500"]):
#     adata.obs["avg_gene_count"] = adata.obs["total_counts"] / adata.obs["n_genes_by_counts"]
#     adata.obs["panel_size"] = label

df = pd.concat([
    adata_140.obs[["avg_gene_count", "panel_size"]],
    adata_500.obs[["avg_gene_count", "panel_size"]],
    adata_1000.obs[["avg_gene_count", "panel_size"]]
])

palette = {"140": "#8ecae6", "500": "#219ebc", "1000": "#023047"}
plt.figure(figsize=(6, 4))
sns.barplot(data=df, x="panel_size", y="avg_gene_count", hue="panel_size", palette=palette, errorbar="sd", legend=False)
sns.despine()
plt.ylabel("Avg counts per gene (per cell)")
plt.xlabel("Panel size")
plt.title("Per-gene averages across panels (common genes only)")
plt.tight_layout()
#plt.savefig(f"{main_dir}methods_images/ev1_panela_inset_all.png", dpi=600, bbox_inches='tight')
plt.show()


## 3. Restore full MERFISH panels before pairwise gene-level comparisons

Do not remove this step: the objects above contain only the 31 genes shared across all three panels.

In [ ]:
adata_140= adata_140_bu.copy()
adata_500= adata_500_bu.copy()
adata_1000= adata_1000_bu.copy()


## 4. Fig. S1D-E — MERFISH 140- versus 500-gene panels

In [ ]:
common_genes= list(set(adata_140.var_names.to_list()) & set(adata_aepd.var_names.to_list()))
adata_140_subset= adata_140[:, common_genes].to_df()
adata_140_md= adata_140.obs.loc[:, ['volume', 'total_counts', 'n_genes_by_counts']]
df_140= pd.concat([adata_140_subset, adata_140_md], axis=1)
df_140['Avg_Count_per_Gene']= df_140['total_counts']/ df_140['n_genes_by_counts']


In [ ]:
adata_aepd_subset= adata_aepd[:, common_genes].to_df()
adata_aepd_md= adata_aepd.obs.loc[:, ['volume', 'total_counts', 'n_genes_by_counts']]
df_aepd= pd.concat([adata_aepd_subset, adata_aepd_md], axis=1)
df_aepd['Avg_Count_per_Gene']= df_aepd['total_counts']/ df_aepd['n_genes_by_counts']


In [ ]:
col_names= ['140_Panel_Avg', '500_Panel_All_Avg']
col_names += ['140_Panel_Rel', '500_Panel_All_Rel']
result_df= pd.DataFrame(index= common_genes, columns= col_names)
for cg in common_genes:
    result_df.loc[cg, '140_Panel_Avg']= np.mean(df_140[cg])
    result_df.loc[cg, '140_Panel_Rel']= np.mean(df_140[cg]/df_140['Avg_Count_per_Gene'])
    result_df.loc[cg, '500_Panel_All_Avg']= np.mean(df_aepd[cg])
    result_df.loc[cg, '500_Panel_All_Rel']= np.mean(df_aepd[cg]/df_aepd['Avg_Count_per_Gene'])


In [ ]:
result_df["140_Panel_Rel"] = pd.to_numeric(result_df["140_Panel_Rel"], errors="coerce")
result_df["500_Panel_All_Rel"] = pd.to_numeric(result_df["500_Panel_All_Rel"], errors="coerce")

result_df["log140"] = np.log10(result_df["140_Panel_Rel"].astype(float) + 1e-6)
result_df["log500"] = np.log10(result_df["500_Panel_All_Rel"].astype(float) + 1e-6)

plot_df = result_df.dropna(subset=["log140", "log500"])

# summary stats
rho, _ = spearmanr(plot_df["log140"], plot_df["log500"])
slope, intercept, r, p, se = linregress(plot_df["log140"], plot_df["log500"])
print(f"Spearman ρ = {rho:.2f}")
print(f"OLS slope = {slope:.2f} ± {1.96*se:.2f}")


In [ ]:
# scatter
ax = sns.regplot(
    data=plot_df, x="log140", y="log500",
    scatter_kws={'s':10, 'alpha':0.5},
    line_kws={'color':'blue'},
)

# Add a label manually to the regression line
line = ax.get_lines()[0]
line.set_label("Regression")
minv, maxv = plot_df[["log140", "log500"]].min().min(), plot_df[["log140", "log500"]].max().max()
plt.plot([minv, maxv], [minv, maxv], "--", color="black", linewidth=1, label="Identity")

plt.xlabel("log₁₀ mean expression (140-gene panel, relative)")
plt.ylabel("log₁₀ mean expression (500-gene panel, relative)")
plt.title(f"Per-gene scaling: ρ={rho:.2f}, slope={slope:.2f}")
sns.despine()
plt.legend(frameon=False)
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panelb_140_500.png", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
plot_df["ratio_drift"] = plot_df["log500"] - plot_df["log140"]

# summary stats
mean_drift = plot_df["ratio_drift"].mean()
median_drift = plot_df["ratio_drift"].median()
iqr_drift = plot_df["ratio_drift"].quantile(0.75) - plot_df["ratio_drift"].quantile(0.25)
print(f"Mean drift = {mean_drift:.3f}, median = {median_drift:.3f}, IQR = {iqr_drift:.3f}")

# histogram
plt.figure(figsize=(5,4))
sns.histplot(plot_df["ratio_drift"], bins=40, color="#219ebc", edgecolor=None)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("log₁₀ ratio (500 / 140)")
plt.ylabel("Gene count")
plt.title("Distribution of per-gene ratio drift")
sns.despine()
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panelc_140_500.png", dpi=600, bbox_inches='tight')
plt.show()


## 5. Fig. 1B and Fig. S1C — MERFISH 500- versus 1000-gene panels

In [ ]:
common_genes= list(set(adata_1000.var_names.to_list()) & set(adata_aepd.var_names.to_list()))
adata_1000_subset= adata_1000[:, common_genes].to_df()
adata_1000_md= adata_1000.obs.loc[:, ['volume', 'total_counts', 'n_genes_by_counts']]
df_1000= pd.concat([adata_1000_subset, adata_1000_md], axis=1)
df_1000['Avg_Count_per_Gene']= df_1000['total_counts']/ df_1000['n_genes_by_counts']
adata_aepd_subset= adata_aepd[:, common_genes].to_df()
adata_aepd_md= adata_aepd.obs.loc[:, ['volume', 'total_counts', 'n_genes_by_counts']]
df_aepd= pd.concat([adata_aepd_subset, adata_aepd_md], axis=1)
df_aepd['Avg_Count_per_Gene']= df_aepd['total_counts']/ df_aepd['n_genes_by_counts']
col_names= ['1000_Panel_Avg', '500_Panel_All_Avg']
col_names += ['1000_Panel_Rel', '500_Panel_All_Rel']
result_df= pd.DataFrame(index= common_genes, columns= col_names)
for cg in common_genes:
    result_df.loc[cg, '1000_Panel_Avg']= np.mean(df_1000[cg])
    result_df.loc[cg, '1000_Panel_Rel']= np.mean(df_1000[cg]/df_1000['Avg_Count_per_Gene'])
    result_df.loc[cg, '500_Panel_All_Avg']= np.mean(df_aepd[cg])
    result_df.loc[cg, '500_Panel_All_Rel']= np.mean(df_aepd[cg]/df_aepd['Avg_Count_per_Gene'])
from scipy.stats import spearmanr, linregress

result_df["1000_Panel_Rel"] = pd.to_numeric(result_df["1000_Panel_Rel"], errors="coerce")
result_df["500_Panel_All_Rel"] = pd.to_numeric(result_df["500_Panel_All_Rel"], errors="coerce")

result_df["log1000"] = np.log10(result_df["1000_Panel_Rel"].astype(float) + 1e-6)
result_df["log500"] = np.log10(result_df["500_Panel_All_Rel"].astype(float) + 1e-6)

plot_df = result_df.dropna(subset=["log1000", "log500"])

# summary stats
rho, _ = spearmanr(plot_df["log500"], plot_df["log1000"])
slope, intercept, r, p, se = linregress(plot_df["log500"], plot_df["log1000"])
print(f"Spearman ρ = {rho:.2f}")
print(f"OLS slope = {slope:.2f} ± {1.96*se:.2f}")


In [ ]:
# scatter
ax = sns.regplot(
    data=plot_df, x="log500", y="log1000",
    scatter_kws={'s':10, 'alpha':0.5},
    line_kws={'color':'blue'},
)

# Add a label manually to the regression line
line = ax.get_lines()[0]
line.set_label("Regression")

minv, maxv = plot_df[["log500", "log1000"]].min().min(), plot_df[["log500", "log1000"]].max().max()
plt.plot([minv, maxv], [minv, maxv], "--", color="black", linewidth=1, label="Identity")

plt.xlabel("log₁₀ mean expression (500-gene panel, relative)")
plt.ylabel("log₁₀ mean expression (1000-gene panel, relative)")
plt.title(f"Per-gene scaling: ρ={rho:.2f}, slope={slope:.2f}")
sns.despine()
plt.legend(frameon=False)
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panelb_500_1000.png", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
plot_df["ratio_drift"] = plot_df["log1000"] - plot_df["log500"]

# summary stats
mean_drift = plot_df["ratio_drift"].mean()
median_drift = plot_df["ratio_drift"].median()
iqr_drift = plot_df["ratio_drift"].quantile(0.75) - plot_df["ratio_drift"].quantile(0.25)
print(f"Mean drift = {mean_drift:.3f}, median = {median_drift:.3f}, IQR = {iqr_drift:.3f}")

# histogram
plt.figure(figsize=(5,4))
sns.histplot(plot_df["ratio_drift"], bins=40, color="#219ebc", edgecolor=None)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("log₁₀ ratio (1000 / 500)")
plt.ylabel("Gene count")
plt.title("Distribution of per-gene ratio drift")
sns.despine()
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panelc_500_1000.png", dpi=600, bbox_inches='tight')
plt.show()


## 6. Load and preprocess external Xenium datasets

Download sources and expected directory contents are documented in the companion README. `xenium_bu` preserves the full 250-gene object before the three-panel common-gene restriction used for Fig. S1B.

In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
replicate_counts = []
replicate_metadata = []

for replicate, dataset_dir in enumerate(xenium_replicate_dirs, start=1):
    counts = sc.read_10x_h5(dataset_dir / 'cell_feature_matrix.h5').to_df()
    metadata = pd.read_parquet(dataset_dir / 'cells.parquet')
    metadata['replicate'] = replicate
    metadata['cell_id'] = f'xm{replicate}_' + metadata['cell_id'].astype(str)
    metadata = metadata.set_index('cell_id')
    metadata.index.name = None

    counts.index = metadata.index
    replicate_counts.append(counts)
    replicate_metadata.append(metadata)

xenium_all = pd.concat(replicate_counts)
xenium_md = pd.concat(replicate_metadata)
xenium_md['replicate'] = pd.Categorical(xenium_md['replicate'])
xenium_data = sc.AnnData(X=xenium_all, obs=xenium_md)

xenium5k = sc.read_10x_h5(xenium_5k_dir / 'cell_feature_matrix.h5')
xenium5k_meta = pd.read_parquet(xenium_5k_dir / 'cells.parquet').set_index('cell_id')
xenium5k_meta.index.name = None
xenium5k.obs = xenium5k_meta.copy()

xenium250 = sc.read_10x_h5(xenium_350_dir / 'cell_feature_matrix.h5')
xenium250_meta = pd.read_parquet(xenium_350_dir / 'cells.parquet').set_index('cell_id')
xenium250_meta.index.name = None
xenium250.obs = xenium250_meta.copy()

xenium_data = qc_and_volume_filter(xenium_data, 'cell_area')
xenium250 = qc_and_volume_filter(xenium250, 'cell_area')
xenium5k = qc_and_volume_filter(xenium5k, 'cell_area')

# Preserve the full 250-gene object before later common-gene subsetting.
xenium_bu = xenium_data.copy()
    ''')
else:
    print('Skipped public Xenium input loading.')


## 7. Fig. 1C and Fig. S1F — MERFISH versus Xenium

In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
common_genes= list(set(adata_pvh.var_names.to_list()) & set(xenium_data.var_names.to_list()))
adata_pvh_subset= adata_pvh[:, common_genes].to_df()
adata_pvh_md= adata_pvh.obs.loc[:, ['volume', 'total_counts', 'n_genes_by_counts']]
df_pvh= pd.concat([adata_pvh_subset, adata_pvh_md], axis=1)
df_pvh['Avg_Count_per_Gene']= df_pvh['total_counts']/ df_pvh['n_genes_by_counts']

xenium_data_subset= xenium_data[:, common_genes].to_df()
xenium_data_md= xenium_data.obs.loc[:, ['cell_area', 'total_counts', 'n_genes_by_counts']]
df_xenium= pd.concat([xenium_data_subset, xenium_data_md], axis=1)
df_xenium['Avg_Count_per_Gene']= df_xenium['total_counts']/ df_xenium['n_genes_by_counts']

col_names= ['MERFISH_Avg', 'Xenium_Avg']
col_names += ['MERFISH_Rel', 'Xenium_Rel']
result_df= pd.DataFrame(index= common_genes, columns= col_names)
for cg in common_genes:
    result_df.loc[cg, 'MERFISH_Avg']= np.mean(df_pvh[cg])
    result_df.loc[cg, 'MERFISH_Rel']= np.mean(df_pvh[cg]/df_pvh['Avg_Count_per_Gene'])
    result_df.loc[cg, 'Xenium_Avg']= np.mean(df_xenium[cg])
    result_df.loc[cg, 'Xenium_Rel']= np.mean(df_xenium[cg]/df_xenium['Avg_Count_per_Gene'])
result_df['MERFISH_Rel'] = pd.to_numeric(result_df['MERFISH_Rel'], errors="coerce")
result_df['Xenium_Rel'] = pd.to_numeric(result_df['Xenium_Rel'], errors="coerce")

result_df["logXenium"] = np.log10(result_df["Xenium_Rel"].astype(float) + 1e-6)
result_df["logMERFISH"] = np.log10(result_df["MERFISH_Rel"].astype(float) + 1e-6)

plot_df = result_df.dropna(subset=["logXenium", "logMERFISH"])

# summary stats
rho, _ = spearmanr(plot_df["logXenium"], plot_df["logMERFISH"])
slope, intercept, r, p, se = linregress(plot_df["logXenium"], plot_df["logMERFISH"])
print(f"Spearman ρ = {rho:.2f}")
print(f"OLS slope = {slope:.2f} ± {1.96*se:.2f}")
    ''')


In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
ax = sns.regplot(
    data=plot_df, x="logXenium", y="logMERFISH",
    scatter_kws={'s':10, 'alpha':0.5},
    line_kws={'color':'blue'},
)

# Add a label manually to the regression line
line = ax.get_lines()[0]
line.set_label("Regression")

minv, maxv = plot_df[["logXenium", "logMERFISH"]].min().min(), plot_df[["logXenium", "logMERFISH"]].max().max()

plt.plot([minv, maxv], [minv, maxv], "--", color="black", linewidth=1, label="Identity")

plt.xlabel("log₁₀ mean expression (Xenium gene panel, relative)")
plt.ylabel("log₁₀ mean expression (MERFISH gene panel, relative)")
plt.title(f"Per-gene scaling: ρ={rho:.2f}, slope={slope:.2f}")
sns.despine()
plt.legend(frameon=False)
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panelc_Xenium_MERFISH.png", dpi=600, bbox_inches='tight')
plt.show()
    ''')


In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
plot_df["ratio_drift"] = plot_df["logMERFISH"] - plot_df["logXenium"]

# summary stats
mean_drift = plot_df["ratio_drift"].mean()
median_drift = plot_df["ratio_drift"].median()
iqr_drift = plot_df["ratio_drift"].quantile(0.75) - plot_df["ratio_drift"].quantile(0.25)
print(f"Mean drift = {mean_drift:.3f}, median = {median_drift:.3f}, IQR = {iqr_drift:.3f}")

# histogram
plt.figure(figsize=(5,4))
sns.histplot(plot_df["ratio_drift"], bins=40, color="#219ebc", edgecolor=None)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("log₁₀ ratio (MERFISH / Xenium)")
plt.ylabel("Gene count")
plt.title("Distribution of per-gene ratio drift")
sns.despine()
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panelc_500_1000.png", dpi=600, bbox_inches='tight')
plt.show()
    ''')


## 8. Fig. 1D — overlap of published brain-aging gene sets

This analysis compares matched oligodendrocyte, endothelial-cell, and microglial gene sets from Tabula Muris Senis, Ximerakis et al., and Jin et al. The external supplementary workbooks are not redistributed here; see [`FIGURE1D_DATA_DOWNLOADS.md`](FIGURE1D_DATA_DOWNLOADS.md) for download and placement instructions.


In [ ]:
aging_sources = AgingGeneSourceFiles(
    tabula_muris_senis=config.aging_gene_dir / "TMS6_DEGs.xlsx",
    ximerakis=config.aging_gene_dir / "PMID_31551601_Differential gene expression data between young and old cell types.xlsx",
    jin_allen=config.aging_gene_dir / "41586_2024_8350_MOESM4_ESM.xlsx",
)

aging_gene_sets, aging_selection_summary = load_figure1d_aging_gene_sets(
    aging_sources,
    adjusted_p_threshold=0.01,
)
display(aging_selection_summary)

figure1d_output_dir = config.results_dir / "figure1d"
figure1d_output_dir.mkdir(parents=True, exist_ok=True)
aging_selection_summary.to_csv(
    figure1d_output_dir / "aging_gene_selection_summary.csv", index=False
)
figure1d_figures = plot_figure1d_venn(
    aging_gene_sets,
    output_dir=figure1d_output_dir,
    dpi=600,
    remove_legend=True,
)


## 9. Fig. S1B — Xenium panel-size comparison

This analysis intentionally subsets all three Xenium objects to the 150 genes they share. The full 250-gene object is restored later before the Xenium–CosMx comparison.

In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
common_genes= list(set(xenium250.var_names.to_list()) & set(xenium_data.var_names.to_list()) & set(xenium5k.var_names.to_list()))
xenium250= xenium250[:, common_genes].copy()
xenium_data= xenium_data[:, common_genes].copy()
xenium5k= xenium5k[:, common_genes].copy()
sc.pp.calculate_qc_metrics(xenium250, qc_vars=(), percent_top=None, log1p=False, inplace=True)
sc.pp.calculate_qc_metrics(xenium_data, qc_vars=(), percent_top=None, log1p=False, inplace=True)
sc.pp.calculate_qc_metrics(xenium5k, qc_vars=(), percent_top=None, log1p=False, inplace=True)

palette = {"350": "#8ecae6", "250": "#219ebc", "5k": "#023047"}

xenium250.obs["panel_size"] = "350"
xenium_data.obs["panel_size"] = "250"
xenium5k.obs["panel_size"] = "5k"
    ''')


In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
# combine into one dataframe
df = pd.concat([
    xenium_data.obs[["total_counts", "panel_size"]],
    xenium250.obs[["total_counts", "panel_size"]],
    xenium5k.obs[["total_counts", "panel_size"]]
])

# basic violin plot
plt.figure(figsize=(6, 4))
sns.violinplot(data=df, x="panel_size", y="total_counts", palette= palette, inner="box", cut=0)
sns.despine()
plt.xlabel("Panel size (number of genes)")
plt.ylabel("Total transcripts per cell")
plt.title("Total counts per cell vs panel size (common genes only)")
# plt.title("Total counts per cell vs panel size")
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panela_all.png", dpi=600, bbox_inches='tight')
plt.show()
    ''')


## 10. Load and preprocess external CosMx data

The analysis retains the original FOV restriction (`fov <= 78`) before QC and cell-area filtering.

In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
cosmx_counts = pd.read_csv(cosmx_dir / '240205104_exprMat_file.csv')
cosmx_meta = pd.read_csv(cosmx_dir / '240205104_metadata_file.csv')

cosmx_counts['cell'] = cosmx_meta['cell']
cosmx_meta = cosmx_meta.set_index('cell')
cosmx_counts = cosmx_counts.set_index('cell')
cosmx_meta.index.name = None
cosmx_counts.index.name = None

cosmx_data = sc.AnnData(X=cosmx_counts, obs=cosmx_meta)
cosmx_data = cosmx_data[cosmx_data.obs['fov'] <= 78].copy()
cosmx_data = qc_and_volume_filter(cosmx_data, 'Area.um2')
    ''')


## 11. Fig. S1G-H — MERFISH versus CosMx

In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
common_genes= list(set(adata_pvh.var_names.to_list()) & set(cosmx_data.var_names.to_list()))
adata_pvh_subset= adata_pvh[:, common_genes].to_df()
adata_pvh_md= adata_pvh.obs.loc[:, ['volume', 'total_counts', 'n_genes_by_counts']]
df_pvh= pd.concat([adata_pvh_subset, adata_pvh_md], axis=1)
df_pvh['Avg_Count_per_Gene']= df_pvh['total_counts']/ df_pvh['n_genes_by_counts']

cosmx_data_subset= cosmx_data[:, common_genes].to_df()
cosmx_data_md= cosmx_data.obs.loc[:, ['Area.um2', 'total_counts', 'n_genes_by_counts']]
df_cosmx= pd.concat([cosmx_data_subset, cosmx_data_md], axis=1)
df_cosmx['Avg_Count_per_Gene']= df_cosmx['total_counts']/ df_cosmx['n_genes_by_counts']

col_names= ['MERFISH_Avg', 'cosmx_Avg']
col_names += ['MERFISH_Rel', 'cosmx_Rel']
result_df= pd.DataFrame(index= common_genes, columns= col_names)
for cg in common_genes:
    result_df.loc[cg, 'MERFISH_Avg']= np.mean(df_pvh[cg])
    result_df.loc[cg, 'MERFISH_Rel']= np.mean(df_pvh[cg]/df_pvh['Avg_Count_per_Gene'])
    result_df.loc[cg, 'cosmx_Avg']= np.mean(df_cosmx[cg])
    result_df.loc[cg, 'cosmx_Rel']= np.mean(df_cosmx[cg]/df_cosmx['Avg_Count_per_Gene'])
result_df['MERFISH_Rel'] = pd.to_numeric(result_df['MERFISH_Rel'], errors="coerce")
result_df['cosmx_Rel'] = pd.to_numeric(result_df['cosmx_Rel'], errors="coerce")

result_df["logcosmx"] = np.log10(result_df["cosmx_Rel"].astype(float) + 1e-6)
result_df["logMERFISH"] = np.log10(result_df["MERFISH_Rel"].astype(float) + 1e-6)

plot_df = result_df.dropna(subset=["logcosmx", "logMERFISH"])

# summary stats
rho, _ = spearmanr(plot_df["logcosmx"], plot_df["logMERFISH"])
slope, intercept, r, p, se = linregress(plot_df["logcosmx"], plot_df["logMERFISH"])
print(f"Spearman ρ = {rho:.2f}")
print(f"OLS slope = {slope:.2f} ± {1.96*se:.2f}")
    ''')


In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
ax = sns.regplot(
    data=plot_df, x="logcosmx", y="logMERFISH",
    scatter_kws={'s':10, 'alpha':0.5},
    line_kws={'color':'blue'},
)

# Add a label manually to the regression line
line = ax.get_lines()[0]
line.set_label("Regression")

minv, maxv = plot_df[["logcosmx", "logMERFISH"]].min().min(), plot_df[["logcosmx", "logMERFISH"]].max().max()

plt.plot([minv, maxv], [minv, maxv], "--", color="black", linewidth=1, label="Identity")


plt.xlabel("log₁₀ mean expression (Cosmx gene panel, relative)")
plt.ylabel("log₁₀ mean expression (MERFISH gene panel, relative)")
plt.title(f"Per-gene scaling: ρ={rho:.2f}, slope={slope:.2f}")
plt.xlim(-1.6,0.2 )
sns.despine()
plt.legend(frameon=False)
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panelc_cosmx_MERFISH.png", dpi=600, bbox_inches='tight')
plt.show()
    ''')


In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
plot_df["ratio_drift"] = plot_df["logMERFISH"] - plot_df["logcosmx"]

# summary stats
mean_drift = plot_df["ratio_drift"].mean()
median_drift = plot_df["ratio_drift"].median()
iqr_drift = plot_df["ratio_drift"].quantile(0.75) - plot_df["ratio_drift"].quantile(0.25)
print(f"Mean drift = {mean_drift:.3f}, median = {median_drift:.3f}, IQR = {iqr_drift:.3f}")

# histogram
plt.figure(figsize=(5,4))
sns.histplot(plot_df["ratio_drift"], bins=40, color="#219ebc", edgecolor=None)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("log₁₀ ratio (MERFISH / Cosmx)")
plt.ylabel("Gene count")
plt.title("Distribution of per-gene ratio drift")
sns.despine()
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panelc_500_1000.png", dpi=600, bbox_inches='tight')
plt.show()
    ''')


## 12. Restore full Xenium data before Xenium–CosMx comparison

Do not remove this step: `xenium_data` was restricted to the genes shared across three Xenium panels in Section 8.

In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    xenium_data=xenium_bu.copy()


## 13. Fig. S1I-J — Xenium versus CosMx

In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
common_genes= list(set(cosmx_data.var_names.to_list()) & set(xenium_data.var_names.to_list()))
cosmx_data_subset= cosmx_data[:, common_genes].to_df()
cosmx_data_md= cosmx_data.obs.loc[:, ['Area.um2', 'total_counts', 'n_genes_by_counts']]
df_cosmx= pd.concat([cosmx_data_subset, cosmx_data_md], axis=1)
df_cosmx['Avg_Count_per_Gene']= df_cosmx['total_counts']/ df_cosmx['n_genes_by_counts']

xenium_data_subset= xenium_data[:, common_genes].to_df()
xenium_data_md= xenium_data.obs.loc[:, ['cell_area', 'total_counts', 'n_genes_by_counts']]
df_xenium= pd.concat([xenium_data_subset, xenium_data_md], axis=1)
df_xenium['Avg_Count_per_Gene']= df_xenium['total_counts']/ df_xenium['n_genes_by_counts']

col_names= ['cosmx_Avg', 'Xenium_Avg']
col_names += ['cosmx_Rel', 'Xenium_Rel']
result_df= pd.DataFrame(index= common_genes, columns= col_names)
for cg in common_genes:
    result_df.loc[cg, 'cosmx_Avg']= np.mean(df_cosmx[cg])
    result_df.loc[cg, 'cosmx_Rel']= np.mean(df_cosmx[cg]/df_cosmx['Avg_Count_per_Gene'])
    result_df.loc[cg, 'Xenium_Avg']= np.mean(df_xenium[cg])
    result_df.loc[cg, 'Xenium_Rel']= np.mean(df_xenium[cg]/df_xenium['Avg_Count_per_Gene'])
result_df['cosmx_Rel'] = pd.to_numeric(result_df['cosmx_Rel'], errors="coerce")
result_df['Xenium_Rel'] = pd.to_numeric(result_df['Xenium_Rel'], errors="coerce")

result_df["logXenium"] = np.log10(result_df["Xenium_Rel"].astype(float) + 1e-6)
result_df["logcosmx"] = np.log10(result_df["cosmx_Rel"].astype(float) + 1e-6)

plot_df = result_df.dropna(subset=["logXenium", "logcosmx"])

# summary stats
rho, _ = spearmanr(plot_df["logXenium"], plot_df["logcosmx"])
slope, intercept, r, p, se = linregress(plot_df["logXenium"], plot_df["logcosmx"])
print(f"Spearman ρ = {rho:.2f}")
print(f"OLS slope = {slope:.2f} ± {1.96*se:.2f}")
ax = sns.regplot(
    data=plot_df, x="logXenium", y="logcosmx",
    scatter_kws={'s':10, 'alpha':0.5},
    line_kws={'color':'blue'},
)

# Add a label manually to the regression line
line = ax.get_lines()[0]
line.set_label("Regression")

minv, maxv = plot_df[["logXenium", "logcosmx"]].min().min(), plot_df[["logXenium", "logcosmx"]].max().max()

plt.plot([minv, maxv], [minv, maxv], "--", color="black", linewidth=1, label="Identity")

print(minv, maxv)
plt.xlabel("log₁₀ mean expression (Xenium gene panel, relative)")
plt.ylabel("log₁₀ mean expression (Cosmx gene panel, relative)")
plt.title(f"Per-gene scaling: ρ={rho:.2f}, slope={slope:.2f}")
sns.despine()
plt.legend(frameon=False)
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panelc_Xenium_cosmx.png", dpi=600, bbox_inches='tight')
plt.show()
plot_df["ratio_drift"] = plot_df["logcosmx"] - plot_df["logXenium"]
    ''')


In [ ]:
if RUN_PUBLIC_PLATFORM_COMPARISONS:
    exec(r'''
# summary stats
mean_drift = plot_df["ratio_drift"].mean()
median_drift = plot_df["ratio_drift"].median()
iqr_drift = plot_df["ratio_drift"].quantile(0.75) - plot_df["ratio_drift"].quantile(0.25)
print(f"Mean drift = {mean_drift:.3f}, median = {median_drift:.3f}, IQR = {iqr_drift:.3f}")

# histogram
plt.figure(figsize=(5,4))
sns.histplot(plot_df["ratio_drift"], bins=40, color="#219ebc", edgecolor=None)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("log₁₀ ratio (cosmx / Xenium)")
plt.ylabel("Gene count")
plt.title("Distribution of per-gene ratio drift")
sns.despine()
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev1_panelc_500_1000.png", dpi=600, bbox_inches='tight')
plt.show()
    ''')


## 14. Fig. 1E-G — matched MERFISH coronal sections

These two sections are loaded from the original Vizgen count and metadata exports, then aligned with the separately generated Allen CCF metadata.

In [ ]:
adata1 = sq.read.vizgen(
    path=vizgen_section1_dir,
    counts_file='cell_by_gene.csv',
    meta_file='cell_metadata.csv',
    transformation_file='micron_to_mosaic_pixel_transform.csv',
    library_id='Yng_1',
)

adata2 = sq.read.vizgen(
    path=vizgen_section2_dir,
    counts_file='cell_by_gene.csv',
    meta_file='cell_metadata.csv',
    transformation_file='micron_to_mosaic_pixel_transform.csv',
    library_id='Yng_2',
)


In [ ]:
def add_ccf_metadata(adata, metadata_file):
    metadata = pd.read_csv(metadata_file, engine='python')
    cell_id_col = metadata.columns[0]
    metadata[cell_id_col] = metadata[cell_id_col].astype(str)
    metadata = metadata.set_index(cell_id_col).reindex(adata.obs_names)

    adata.obs['Region'] = metadata['Brain Region ID'].values
    adata.obs['Region Name'] = pd.Categorical(metadata['Brain Region Name'])
    return adata


adata1 = add_ccf_metadata(
    adata1,
    ccf_metadata_dir / 'cell_metadata_wCCF_regions_young_anterior_section_example_1.csv',
)
adata2 = add_ccf_metadata(
    adata2,
    ccf_metadata_dir / 'cell_metadata_wCCF_regions_young_anterior_section_example_2.csv',
)

adata1 = adata1[adata1.obs['Region Name'].notna()].copy()
adata2 = adata2[adata2.obs['Region Name'].notna()].copy()

for adata in (adata1, adata2):
    adata.obs['Regions'] = pd.Categorical(
        adata.obs['Region Name'].astype(str).str.replace(r',.*', '', regex=True)
    )
    sc.pp.calculate_qc_metrics(
        adata,
        qc_vars=(),
        percent_top=None,
        log1p=False,
        inplace=True,
    )


In [ ]:
from matplotlib.colors import LogNorm

sq.pl.spatial_scatter(
    adata1,
    color="total_counts",
    cmap="inferno",
    norm=LogNorm(vmin=50, vmax=4000),
    shape=None
)

# plt.savefig(f"{main_dir}methods_images/adata1_spatial_total_counts_log_v.png", dpi=600, bbox_inches='tight')


In [ ]:
sq.pl.spatial_scatter(
    adata2,
    color="total_counts",
    cmap="inferno",
    norm=LogNorm(vmin=50, vmax=4000),
    shape=None
)
# plt.savefig(f"{main_dir}methods_images/adata2_spatial_total_counts_log_v.png", dpi=600, bbox_inches='tight')


In [ ]:
sq.pl.spatial_scatter(
    adata1,
    color=[
        'Regions'
    ],
    shape=None,

)


In [ ]:
sq.pl.spatial_scatter(
    adata2,
    color=[
        'Regions'
    ],
    shape=None,

)


In [ ]:
adata1_str= adata1[adata1.obs['Regions'].isin(['Caudoputamen','Nucleus accumbens', 'anterior commissure'])].copy()
adata2_str= adata2[adata2.obs['Regions'].isin(['Caudoputamen','Nucleus accumbens', 'anterior commissure'])].copy()


In [ ]:
sq.pl.spatial_scatter(
    adata1_str,
    color="total_counts",
    cmap="inferno",
    norm=LogNorm(vmin=50, vmax=4000),
    shape=None
)

# plt.savefig(f"{main_dir}methods_images/adata1_spatial_region_ss.png", dpi=600, bbox_inches='tight')



sq.pl.spatial_scatter(
    adata2_str,
    color="total_counts",
    cmap="inferno",
    norm=LogNorm(vmin=50, vmax=4000),
    shape=None
)
# plt.savefig(f"{main_dir}methods_images/adata2_spatial_region_ss.png", dpi=600, bbox_inches='tight')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors

def local_depth_profile(adata, sample_n=3000, n_neighbors=15, normalize=True):
    # --- use all cells to build the neighbor graph ---
    coords_full = adata.obsm["spatial"]
    counts_full = adata.obs["total_counts"].values.astype(float)

    # optional per-slide normalization (keeps units comparable across slides)
    if normalize:
        counts_full = counts_full / np.median(counts_full)

    # fit KNN on the FULL coordinate set
    nbrs = NearestNeighbors(n_neighbors=n_neighbors+1, algorithm="kd_tree")
    nbrs.fit(coords_full)
    dists_full, nbr_idx_full = nbrs.kneighbors(coords_full)  # shape: (N, k+1)
    dists_full = dists_full[:, 1:]       # drop self
    nbr_idx_full = nbr_idx_full[:, 1:]

    # now sample focal cells for efficiency (neighbors remain from full data)
    n = min(sample_n, adata.n_obs)
    rng = np.random.default_rng(0)
    focal = rng.choice(adata.n_obs, n, replace=False)

    dists = dists_full[focal]                          # (n, k)
    diffs = np.abs(counts_full[focal, None] - counts_full[nbr_idx_full[focal]])  # (n, k)

    # bin distances up to (say) the 95th percentile of neighbor distances
    dmax = np.percentile(dists, 95)
    bins = np.linspace(0, dmax, 30)
    centers = 0.5 * (bins[:-1] + bins[1:])
    means = []
    for i in range(len(bins)-1):
        mask = (dists >= bins[i]) & (dists < bins[i+1])
        if np.any(mask):
            means.append(diffs[mask].mean())
        else:
            means.append(np.nan)
    return centers, np.array(means)

# example: overlay two slides
x1, y1 = local_depth_profile(adata1, sample_n=3000, n_neighbors=15, normalize=False)
x2, y2 = local_depth_profile(adata2, sample_n=3000, n_neighbors=15, normalize=False)

plt.figure(figsize=(6,4))
plt.plot(x1, y1, marker="o", lw=1.5, label="Slide 1")
plt.plot(x2, y2, marker="o", lw=1.5, label="Slide 2")
plt.xlabel("Neighbor distance")
plt.ylabel("Mean |Δ normalized total_counts|")
plt.title("Local depth smoothness across slides")
plt.legend(frameon=False)
plt.tight_layout()
# plt.savefig(f"{main_dir}methods_images/ev2_panelc.png", dpi=600, bbox_inches='tight')

plt.show()
